# Spoof / smoke / clock / OTR — Pass 1

**Deck:** Filimonov (Perm Winter School 2013) slides **29–30** (smoking/layering), **35–37** (clock hunting), **41–43** (OTR policy).

**Slice:** ETH · Hyperliquid + Deribit + Kraken · UTC days `2026-09-26`, `2026-09-27`, `2026-09-30`.

**Lib:** `research.lib.hftpat` — `smoke_spoof_proxy`, `clock_cluster_*`, `otr_aggregate`.

**Bias:** Hold/Kill. Frank FP rates (time-shift placebo). OTR = venue policy-monitor only — **no firm IDs**.

Artifacts: `out/spoof_smoke_clock/` · regenerate via `scripts/exp_spoof_clock.py`.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display, Markdown

BOOK = Path(os.environ.get("ARES_MICROSTRUCTURE") or (Path.home() / "srv" / "ares-microstructure")) / 'research' / 'books' / 'filmonov'
OUT = BOOK / 'out' / 'spoof_smoke_clock'
FIGS = OUT / 'figs'

payload = json.loads((OUT / 'pass1.json').read_text())
summary = payload['summary']
rows = payload['rows']
comp = pd.DataFrame(json.loads((OUT / 'completeness.json').read_text()))

print('symbol', payload['symbol'], 'days', payload['days'])
print('n_ok', summary['n_ok'], 'n_complete', summary['n_complete'], 'n_trade_synth', summary['n_trade_synth'])
print('mean smoke FP contam', round(summary['mean_smoke_fp_contam'], 3))
print('mean clock max_z', round(summary['mean_clock_max_z'], 2), 'mean n_excess', round(summary['mean_clock_n_excess'], 2))
print('mean otr_day', round(summary['mean_otr_day'], 3))
print('bias:', summary['bias'])
display(comp)

## Smoke / layer FP board

When `n_smoke>0`, time-shift placebo contamination averages **≈1.8** (placebo ≥ observed → surplus share 0). **Kill** as detector.

In [ ]:
smoke_rows = []
for r in rows:
    if not r.get('ok'):
        continue
    s = r['smoke']
    smoke_rows.append({
        'venue': r['venue'],
        'day': r['day'],
        'n_smoke': s['n_smoke'],
        'n_layer': s['n_layer'],
        'hit_rate': s['smoke_hit_rate'],
        'fp_contam': s['smoke_fp_contam'],
        'surplus': s['smoke_surplus_share'],
        'synth': r.get('is_trade_synth'),
    })
display(pd.DataFrame(smoke_rows))
display(Image(filename=str(FIGS / 'fig_smoke_fp_rates.png')))
display(Image(filename=str(FIGS / 'fig_completeness_fp.png')))

## Clock cluster (second-of-minute)

Uniform-null excess is large; shuffled within-minute placebo has **0** excess bins. **Hold** as algo-hunter monitor.

In [ ]:
clock_rows = [{
    'venue': r['venue'],
    'day': r['day'],
    'max_z': r['clock']['max_z'],
    'n_excess': r['clock']['n_excess'],
    'placebo_n_excess': r['clock']['placebo_n_excess'],
    'excess_vs_placebo': r['clock']['excess_vs_placebo'],
} for r in rows if r.get('ok')]
display(pd.DataFrame(clock_rows))
display(Image(filename=str(FIGS / 'fig_clock_cluster.png')))

## Venue OTR (policy-only)

Cancel-proxy / trades at day grain. Kraken `trade_synth` → OTR=0. **Kill** participant-level OTR vanity.

In [ ]:
otr_rows = [{
    'venue': r['venue'],
    'day': r['day'],
    'otr_day': r['otr']['otr_day'],
    'otr_mean_1s': r['otr']['otr_mean_1s'],
    'n_cancel': r['otr']['n_cancel_day'],
    'n_trade': r['otr']['n_trade_day'],
    'policy_only': r['otr']['policy_only'],
    'synth': r.get('is_trade_synth'),
} for r in rows if r.get('ok')]
display(pd.DataFrame(otr_rows))
display(Image(filename=str(FIGS / 'fig_otr_regimes.png')))
Markdown(payload.get('decisions_draft', {}).__repr__())

## Decisions (Pass 1)

| id | decision |
|----|----------|
| `spoof.smoke_proxy` | **Kill** (FP contam ≈1.8 when firing) |
| `spoof.layer_proxy` | **Kill** as α |
| `spoof.clock_cluster` | **Hold** monitor |
| `spoof.otr_venue` | **Hold** policy-only; Kill firm-ID OTR |

See [`CANDIDATES.md`](CANDIDATES.md) · [`EXP_REPORT.md`](EXP_REPORT.md).